# RUDRA - Physics-Informed Turbomachine Prognostics (full N-CMAPSS dataset)
### Component Health Indices & Remaining Useful Life - all 10 datasets (DS01 to DS08c)

---
### Upgraded Production Prognostics Engine Architecture:
- **PyTorch Squeeze-and-Excitation Residual Core (SE-ResNet-MLP)**: Deep continuous degradation model with channel attention, LayerNorm, and SiLU activations dynamically reweighting features as components age.
- **Simplex-Constrained Convex Ensembling**: Non-Negative Least Squares (NNLS) meta-learner guaranteeing positive, non-canceling ensemble weights across LightGBM, CatBoost, XGBoost, Random Forest, and PyTorch SE-ANN.
- **First-Principles Aerothermal Station Thermodynamics**: Explicit Fan pressure ratios ($P_{15}/P_2, P_{21}/P_2$), isentropic compressor and turbine efficiencies, and aerodynamic loading proxies that directly resolve Fan and Compressor health.
- **Unit-Specific Initial Calibration**: Removes engine-to-engine manufacturing variation by referencing the first 5 healthy flights for all primary stations.
- **Monotonic Health Degradation Envelopes**: Physics-informed monotonic health tracking preventing non-physical component self-healing mid-flight.
- **Dynamic Innovation-Gated Dual Kalman Filter**: Velocity-coupled process noise covariance $Q(t)$ snaps to sudden degradation acceleration with zero phase lag, paired with an offline Rauch-Tung-Striebel (RTS) zero-phase smoother.
- **Zero Cycle-Counter Leakage**: Strictly prognostic physical modeling without countdown shortcuts.


In [ ]:
# -- Cell 1: Package Installation, Deep Neural Core & Serializable Estimators --
import os, subprocess, sys

os.environ.setdefault('KAGGLE_USERNAME', 'placeholder')
os.environ.setdefault('KAGGLE_KEY', 'placeholder')

def pip(*pkgs): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

for mod, pkg in [('lightgbm','lightgbm'), ('xgboost','xgboost'), ('catboost','catboost'),
                  ('shap','shap'), ('h5py','h5py'), ('kaggle','kaggle'), ('kagglehub','kagglehub'),
                  ('torch','torch'), ('scipy','scipy')]:
    try: __import__(mod)
    except Exception: pip(pkg)

import lightgbm as lgb
import catboost as cb; from catboost import CatBoostRegressor
import xgboost as xgb
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.optimize import minimize
import numpy as np, pandas as pd, h5py, os, json, joblib, warnings, shutil, time
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

# --- Squeeze-and-Excitation (SE) Channel-Attention Residual Core ---
class SqueezeExcitation(nn.Module):
    """Channel attention mechanism that dynamically weights degradation features as the engine ages."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 8), bias=False),
            nn.SiLU(),
            nn.Linear(max(channels // reduction, 8), channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x)

class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.act = nn.SiLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.se  = SqueezeExcitation(hidden_dim)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        res = x
        out = self.act(self.ln1(self.fc1(x)))
        out = self.drop(out)
        out = self.ln2(self.fc2(out))
        out = self.se(out)
        return self.act(out + res)

class ResidualANN(nn.Module):
    def __init__(self, in_features, hidden_dim=128, n_blocks=2, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout)
        )
        self.blocks = nn.ModuleList([ResidualBlock(hidden_dim, dropout=dropout) for _ in range(n_blocks)])
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
    def forward(self, x):
        h = self.in_proj(x)
        for b in self.blocks: h = b(h)
        return self.head(h).squeeze(-1)

class ResidualANNRegressor:
    """Deep Residual ANN with Squeeze-and-Excitation, LR warmup, cosine decay, and sample-weighting."""
    def __init__(self, hidden_dim=128, n_blocks=2, lr=1.5e-3, epochs=100,
                 patience=18, min_epochs=35, val_frac=0.12,
                 batch_size=128, dropout=0.1, random_state=42):
        self.hidden_dim, self.n_blocks, self.lr = hidden_dim, n_blocks, lr
        self.epochs, self.patience, self.min_epochs = epochs, patience, min_epochs
        self.val_frac   = val_frac
        self.batch_size = batch_size
        self.dropout    = dropout
        self.random_state = random_state
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model  = None
        self.scaler_mean, self.scaler_std = None, None

    def _make_loader(self, X_norm, y, sw, shuffle):
        ds = TensorDataset(torch.from_numpy(X_norm.astype(np.float32)),
                           torch.from_numpy(y.astype(np.float32)),
                           torch.from_numpy(sw.astype(np.float32)))
        return DataLoader(ds, batch_size=self.batch_size,
                          shuffle=shuffle, drop_last=False)

    def _epoch_loss(self, loader, criterion):
        self.model.eval()
        total, n = 0.0, 0
        with torch.no_grad():
            for bx, by, bw in loader:
                bx, by, bw = bx.to(self.device), by.to(self.device), bw.to(self.device)
                loss = (criterion(self.model(bx), by) * bw).mean().item()
                total += loss * len(bx); n += len(bx)
        return total / max(n, 1)

    def fit(self, X, y, sample_weight=None):
        import copy as _copy
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)

        self.scaler_mean = np.mean(X, axis=0, keepdims=True)
        self.scaler_std  = np.std(X,  axis=0, keepdims=True) + 1e-6
        X_norm = (X - self.scaler_mean) / self.scaler_std

        sw = np.ones(len(y), dtype=np.float32) if sample_weight is None else np.asarray(sample_weight, dtype=np.float32)
        sw = sw / (np.mean(sw) + 1e-6)

        n_val = max(1, int(len(y) * self.val_frac))
        idx   = np.argsort(y)
        val_idx  = idx[::max(1, len(idx)//n_val)][:n_val]
        train_mask = np.ones(len(y), bool); train_mask[val_idx] = False
        trn_idx = np.where(train_mask)[0]

        trn_loader = self._make_loader(X_norm[trn_idx], y[trn_idx], sw[trn_idx], True)
        val_loader = self._make_loader(X_norm[val_idx], y[val_idx], sw[val_idx], False)

        in_dim = X.shape[1]
        self.model = ResidualANN(in_dim, self.hidden_dim, self.n_blocks, self.dropout).to(self.device)
        optimizer = optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-4)

        warmup  = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=5)
        cosine  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, self.epochs - 5), eta_min=1e-5)
        scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

        criterion  = nn.SmoothL1Loss(reduction='none', beta=1.0)
        best_val   = float('inf')
        best_state = _copy.deepcopy(self.model.state_dict())
        no_improve = 0

        for epoch in range(self.epochs):
            self.model.train()
            for bx, by, bw in trn_loader:
                bx, by, bw = bx.to(self.device), by.to(self.device), bw.to(self.device)
                optimizer.zero_grad()
                loss = (criterion(self.model(bx), by) * bw).mean()
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=2.0)
                optimizer.step()
            scheduler.step()

            val_loss = self._epoch_loss(val_loader, criterion)
            if val_loss < best_val - 1e-5:
                best_val   = val_loss
                best_state = _copy.deepcopy(self.model.state_dict())
                no_improve = 0
            else:
                no_improve += 1

            if epoch >= self.min_epochs and no_improve >= self.patience:
                break

        self.model.load_state_dict(best_state)
        return self

    def predict(self, X):
        self.model.eval()
        X_norm = (X - self.scaler_mean) / self.scaler_std
        with torch.no_grad():
            t_x = torch.from_numpy(X_norm.astype(np.float32)).to(self.device)
            preds = self.model(t_x).cpu().numpy()
        return preds

    def save_model(self, path):
        torch.save({
            'state_dict': self.model.state_dict(),
            'scaler_mean': self.scaler_mean,
            'scaler_std':  self.scaler_std,
            'hidden_dim':  self.hidden_dim,
            'n_blocks':    self.n_blocks,
            'dropout':     self.dropout
        }, path)

# --- Non-Negative Simplex Constrained Meta-Learner (Convex Combination) ---
class SimplexConstrainedMetaLearner:
    """Non-Negative Simplex Constrained Meta-Learner.
    Enforces weights >= 0 and sum(weights) == 1.0, preventing collinear base models
    from canceling each other out and guaranteeing positive, stable ensembling.
    """
    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept
        self.coef_ = None
        self.intercept_ = 0.0

    def fit(self, X, y, sample_weight=None):
        n_samples, n_features = X.shape
        sw = np.ones(n_samples) if sample_weight is None else np.asarray(sample_weight, dtype=float)
        sw = sw / (np.mean(sw) + 1e-6)

        if self.fit_intercept:
            def loss_func(params):
                w = params[:-1]
                b = params[-1]
                pred = X @ w + b
                return np.mean(sw * (pred - y)**2)
            bounds = [(0.0, 1.0) for _ in range(n_features)] + [(-15.0, 15.0)]
            constraints = ({'type': 'eq', 'fun': lambda p: np.sum(p[:-1]) - 1.0})
            init_params = np.append(np.ones(n_features) / n_features, float(np.mean(y) - np.mean(X)))
            res = minimize(loss_func, init_params, bounds=bounds, constraints=constraints, method='SLSQP')
            self.coef_ = np.maximum(res.x[:-1], 0.0)
            self.coef_ = self.coef_ / np.sum(self.coef_)
            self.intercept_ = float(res.x[-1])
        else:
            def loss_func(w):
                pred = X @ w
                return np.mean(sw * (pred - y)**2)
            bounds = [(0.0, 1.0) for _ in range(n_features)]
            constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})
            init_w = np.ones(n_features) / n_features
            res = minimize(loss_func, init_w, bounds=bounds, constraints=constraints, method='SLSQP')
            self.coef_ = np.maximum(res.x, 0.0)
            self.coef_ = self.coef_ / np.sum(self.coef_)
            self.intercept_ = 0.0
        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_

class ZeroWearIdentityMeta:
    def __init__(self, val=0.0, n_models=5):
        self.val = float(val)
        self.coef_ = np.zeros(n_models, dtype=float)
        self.intercept_ = float(val)
    def predict(self, X):
        return np.full(len(X), self.val, dtype=float)

class ZeroWearIdentityModel:
    def __init__(self, val=0.0):
        self.val = float(val)
    def fit(self, X, y, sample_weight=None): return self
    def predict(self, X): return np.full(len(X), self.val, dtype=float)
    def save_model(self, path, *args, **kwargs):
        with open(path, 'w') as f: json.dump({'constant': self.val}, f)

print('Environment Ready:')
print(f'   LightGBM {lgb.__version__} | XGBoost {xgb.__version__} | CatBoost {cb.__version__} | PyTorch {torch.__version__} (Device: {torch.device("cuda" if torch.cuda.is_available() else "cpu")})')


In [ ]:
# -- Cell 2: Kaggle credentials - upload kaggle.json ------------------------
import os, sys, json, base64, urllib.request, urllib.error

IN_COLAB = 'google.colab' in sys.modules
DATASET_SLUG = 'bishals098/nasa-cmapss-2-engine-degradation'
# The 9 fully verified NASA N-CMAPSS subsets available on the Kaggle mirror (~27 GB raw).
# Note: DS08d-010 is excluded by default as its Kaggle mirror upload was truncated (internal HDF5 heap error).
ALL_FILES = ['N-CMAPSS_DS01-005.h5', 'N-CMAPSS_DS02-006.h5', 'N-CMAPSS_DS03-012.h5', 'N-CMAPSS_DS04.h5',
             'N-CMAPSS_DS05.h5', 'N-CMAPSS_DS06.h5', 'N-CMAPSS_DS07.h5', 'N-CMAPSS_DS08a-009.h5',
             'N-CMAPSS_DS08c-008.h5']
# Default is all 9 verified files; set e.g. RUDRA_FILES="N-CMAPSS_DS02-006.h5" to select specific subsets.
FILES = [f.strip() for f in os.environ.get('RUDRA_FILES', ','.join(ALL_FILES)).split(',') if f.strip()]
DATA_DIR = os.environ.get('RUDRA_DATA_DIR', '/content/ncmapss')     # local copies here are used instead of downloading
os.makedirs(DATA_DIR, exist_ok=True)

KAGGLE_DIR = os.path.expanduser('~/.kaggle')
KAGGLE_JSON_PATH = os.path.join(KAGGLE_DIR, 'kaggle.json')

def install_kaggle_json(raw):
    creds = json.loads(raw.decode('utf-8') if isinstance(raw, bytes) else raw)
    if not isinstance(creds, dict) or not {'username', 'key'} <= set(creds):
        raise ValueError("This is not a kaggle.json file: it must contain 'username' and 'key'. "
                         "Create one at kaggle.com > Settings > API > Create New Token.")
    os.makedirs(KAGGLE_DIR, exist_ok=True)
    with open(KAGGLE_JSON_PATH, 'w') as f:
        json.dump({'username': creds['username'], 'key': creds['key']}, f)
    os.chmod(KAGGLE_JSON_PATH, 0o600)
    return creds

def check_credentials(creds):
    """Ask Kaggle for the dataset's file list with these credentials; a 401/403 means a wrong or expired token."""
    req = urllib.request.Request(f'https://www.kaggle.com/api/v1/datasets/list/{DATASET_SLUG}')
    token = base64.b64encode(f"{creds['username']}:{creds['key']}".encode()).decode()
    req.add_header('Authorization', f'Basic {token}')
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            names = [f['name'] for f in json.load(r).get('datasetFiles', [])]
        missing = [f for f in FILES if f not in names]
        print(f'Kaggle credentials OK (user {creds["username"]}); dataset lists {len(names)} files.'
              + (f' NOT listed: {missing}' if missing else ''))
    except urllib.error.HTTPError as e:
        if e.code in (401, 403):
            raise RuntimeError(f'Kaggle rejected kaggle.json for user {creds["username"]} (HTTP {e.code}). The token is '
                               'wrong or expired: create a new one (kaggle.com > Settings > API > Create New Token), '
                               'delete ~/.kaggle/kaggle.json and re-run this cell.') from e
        print(f'Could not verify the credentials (HTTP {e.code}); continuing.')
    except Exception as e:
        print(f'Could not reach Kaggle to verify the credentials ({e!r}); continuing.')

all_local = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in FILES)
if all_local:
    print(f'All {len(FILES)} file(s) already in {DATA_DIR}: no Kaggle download needed.')
else:
    has_valid_json = False
    if os.path.exists(KAGGLE_JSON_PATH):
        try:
            with open(KAGGLE_JSON_PATH) as f:
                c_saved = json.load(f)
            if c_saved.get('username') not in ('placeholder', '', None) and c_saved.get('key') not in ('placeholder', '', None):
                has_valid_json = True
        except Exception:
            pass

    if has_valid_json:
        with open(KAGGLE_JSON_PATH) as f:
            creds = install_kaggle_json(f.read())
        print(f'Using existing {KAGGLE_JSON_PATH}.')
    elif IN_COLAB:
        from google.colab import files as colab_files
        print('Upload your kaggle.json (kaggle.com > Settings > API > Create New Token):')
        uploaded = colab_files.upload()
        if not uploaded:
            raise RuntimeError('No file uploaded. Re-run this cell and choose kaggle.json.')
        name = next((n for n in uploaded if n.lower().endswith('.json')), next(iter(uploaded)))
        creds = install_kaggle_json(uploaded[name])
        print(f'Saved {name} as {KAGGLE_JSON_PATH}.')
    else:
        raise RuntimeError(f'Put kaggle.json in {KAGGLE_DIR} (or the .h5 files in {DATA_DIR}) and re-run.')
    os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = creds['username'], creds['key']
    if 'kaggle' in sys.modules:
        try:
            import kaggle
            kaggle.api.authenticate()
        except Exception:
            pass
    check_credentials(creds)
print(f'{len(FILES)} file(s) to process: {FILES}')

In [ ]:
# -- Cell 2b (optional): Google Drive cache of the aggregated flight table ----
# The download + aggregation of all files takes a while; with Drive mounted the aggregated table is saved and a
# re-run (e.g. after a Colab disconnect) loads it in seconds. Set USE_DRIVE_CACHE = False to skip.
USE_DRIVE_CACHE = True
CACHE_FILE = None
if USE_DRIVE_CACHE and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs('/content/drive/MyDrive/rudra_ncmapss', exist_ok=True)
        CACHE_FILE = '/content/drive/MyDrive/rudra_ncmapss/flights_' + '_'.join(
            f.replace('N-CMAPSS_', '').replace('.h5', '') for f in FILES) + '.pkl'
    except Exception as e:
        print('Drive not mounted, no cache:', repr(e))
print('aggregate cache:', CACHE_FILE or 'none')

In [ ]:
# -- Cell 3: Download helpers, truncated-file repair, HDF5 inspection --------
import glob, re, shutil, subprocess, zipfile, h5py

def find_local(fname):
    hits = glob.glob(os.path.join(DATA_DIR, '**', fname), recursive=True)
    return hits[0] if hits else None

def download(fname):
    """(path, downloaded). kagglehub first (uses kaggle.json / KAGGLE_* env); Kaggle CLI as fallback (+ unzip)."""
    local = find_local(fname)
    if local:
        return local, False
    try:
        import kagglehub
        return kagglehub.dataset_download(DATASET_SLUG, path=fname), True
    except Exception as e:
        print(f'  kagglehub failed ({e!r}); trying the Kaggle CLI...')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', DATASET_SLUG, '-f', fname, '-p', DATA_DIR], check=True)
    z = os.path.join(DATA_DIR, fname + '.zip')
    if os.path.exists(z):                                   # single-file downloads can arrive zipped
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA_DIR)
        os.remove(z)
    path = find_local(fname)
    if not path:
        raise FileNotFoundError(f'{fname} was not found after downloading; check the Kaggle output above.')
    return path, True

def open_or_repair(path, workdir='/tmp'):
    """Tests opening the HDF5 file. If truncated, attempts zero-padding up to the recorded size (at most 4 KB).
    Verifies that the file's internal heap and group links are valid without raising HDF5 free-block errors.
    Returns (path to open, repaired copy to delete or None). Returns (None, None) if file is unrecoverable."""
    try:
        with h5py.File(path, 'r') as hf:
            _ = list(hf.keys())
        return path, None
    except (OSError, RuntimeError) as exc:
        m = re.search(r'eof = (\d+), sblock->base_addr = \d+, stored_eof = (\d+)', str(exc))
        if not m or not 0 < int(m.group(2)) - int(m.group(1)) <= 4096:
            print(f'  WARNING: {os.path.basename(path)} has unrecoverable HDF5 corruption ({exc}); skipping.')
            return None, None
        missing = int(m.group(2)) - int(m.group(1))
    os.makedirs(workdir, exist_ok=True)
    fixed = os.path.join(workdir, 'repaired_' + os.path.basename(path))
    shutil.copyfile(path, fixed)
    with open(fixed, 'ab') as f:
        f.write(b'\0' * missing)
    try:
        with h5py.File(fixed, 'r') as hf:
            _ = list(hf.keys())
        print(f'  NOTE: {os.path.basename(path)} was {missing} bytes short; using zero-padded copy')
        return fixed, fixed
    except Exception as exc:
        print(f'  WARNING: {os.path.basename(path)} remains structurally corrupt after padding ({exc}); skipping.')
        try: os.remove(fixed)
        except OSError: pass
        return None, None

def inspect_h5(path):
    with h5py.File(path, 'r') as hf:
        print('Available HDF5 keys:')
        for k in sorted(hf.keys()):
            print(f'  {k:12s} shape={hf[k].shape} dtype={hf[k].dtype}')
        if 'T_var' in hf:
            print('  T_var:', [str(s).strip() for s in hf['T_var'][:].astype('U32').ravel()])

In [ ]:
# -- Cell 4: Download + ingestion + per-cycle aggregation, file by file --------
XS_COLS = ['T24','T30','T48','T50','P15','P2','P21','P24','Ps30','P40','P50','Nf','Nc','Wf']
W_COLS  = ['alt','Mach','TRA','T2']
T_DEV_COLS = [
    'fan_eff_mod','fan_flow_mod',
    'LPC_eff_mod','LPC_flow_mod',
    'HPC_eff_mod','HPC_flow_mod',
    'HPT_eff_mod','HPT_flow_mod',
    'LPT_eff_mod','LPT_flow_mod',
]
A_COLS = ['unit', 'cycle', 'Fc', 'hs']

STANDARD_VARS = {'W_var': W_COLS, 'X_s_var': XS_COLS, 'T_var': T_DEV_COLS, 'A_var': A_COLS}

def var_names(hf, key):
    """Column names stored in the file. If they are blank or damaged (e.g. bytes lost in a truncated download), fall
    back to the standard N-CMAPSS order - only when the column count matches - and say so."""
    names = [str(s).strip() for s in hf[key][:].astype('U32').ravel()]
    std = STANDARD_VARS.get(key)
    if std and (not all(names) or len(set(names) & set(std)) < len(std) // 2):
        n_cols = hf[key.replace('_var', '_dev')].shape[1] if key.replace('_var', '_dev') in hf else len(names)
        if n_cols != len(std):
            raise ValueError(f'{key} is unreadable and has {n_cols} columns (standard: {len(std)})')
        print(f'  WARNING: {key} names unreadable {names}; using the standard N-CMAPSS order')
        return list(std)
    return names

def load_split(h5_path, split):
    """One split of one file -> per-cycle medians, with every column mapped BY NAME from the *_var keys."""
    if not h5_path:
        return None
    try:
        with h5py.File(h5_path, 'r') as hf:
            try:
                _ = hf[f'X_s_{split}']
            except (KeyError, RuntimeError, OSError):
                return None
            parts = {}
            for grp in ('W', 'X_s', 'T', 'A'):
                names = var_names(hf, f'{grp}_var')
                arr = hf[f'{grp}_{split}'][:].astype(np.float32)
                parts[grp] = pd.DataFrame(arr, columns=names)
            y = hf[f'Y_{split}'][:].astype(np.float32).ravel()
    except (RuntimeError, OSError, Exception) as exc:
        print(f'  WARNING: Failed reading {split} from {os.path.basename(h5_path)}: {exc}; skipping split.')
        return None

    n = {k: len(v) for k, v in parts.items()}
    if len(set(n.values()) | {len(y)}) != 1:
        print(f'  WARNING: {h5_path} [{split}]: groups not row-aligned: {n}, Y={len(y)}; skipping split.')
        return None
    for need, grp in ((W_COLS, 'W'), (XS_COLS, 'X_s'), (A_COLS, 'A')):
        missing = [c for c in need if c not in parts[grp]]
        if missing:
            print(f'  WARNING: {h5_path} [{split}]: missing {grp} variables {missing}; skipping split.')
            return None
    used_t = [c for c in T_DEV_COLS if c in parts['T']]
    df = pd.concat([parts['A'][A_COLS].rename(columns={'Fc': 'flight_class'}), parts['W'][W_COLS],
                    parts['X_s'][XS_COLS], parts['T'][used_t]], axis=1)
    df['RUL'] = y
    del parts
    agg = df.groupby(['unit','cycle']).median().reset_index()
    del df
    agg['unit'] = agg['unit'].astype(int)
    agg['cycle'] = agg['cycle'].astype(int)
    agg['hs'] = agg['hs'].round().astype(int)
    agg['RUL'] = agg['RUL'].astype(float)
    return agg.sort_values(['unit','cycle']).reset_index(drop=True)

flights = None
if CACHE_FILE and os.path.exists(CACHE_FILE):
    flights = pd.read_pickle(CACHE_FILE)
    print(f'Loaded the aggregated table from the Drive cache ({len(flights):,} flight cycles).')
if flights is None:
    parts = []
    for fi, fname in enumerate(FILES):
        t0 = time.time()
        try:
            path, downloaded = download(fname)
        except Exception as exc:
            print(f'  WARNING: Failed downloading {fname}: {exc}; skipping.')
            continue
        h5_path, repaired = open_or_repair(path)
        if h5_path is None:
            if downloaded:
                try: os.remove(path)
                except OSError: pass
            continue
        if fi == 0 or len(parts) == 0:
            inspect_h5(h5_path)
        ds = fname.replace('N-CMAPSS_', '').replace('.h5', '')
        for split in ('dev', 'test'):
            try:
                agg = load_split(h5_path, split)
            except Exception as exc:
                print(f'  WARNING: Failed parsing {fname} [{split}]: {exc}; skipping split.')
                agg = None
            if agg is None:
                continue
            agg.insert(0, 'dataset', ds)
            agg.insert(1, 'split', split)
            agg.insert(2, 'unit_local', agg['unit'])
            agg['unit'] = (fi + 1) * 1000 + agg['unit']           # unit ids repeat across files -> make them unique
            parts.append(agg)
            print(f'  {fname} [{split}]: {agg["unit"].nunique()} engines, {len(agg):,} flight cycles')
        if repaired:
            try: os.remove(repaired)
            except OSError: pass
        if downloaded:                                            # free the disk before the next file
            try:
                os.remove(path)
            except OSError:
                pass
        gc.collect() if 'gc' in globals() else None
        print(f'{fname} done in {time.time() - t0:.0f}s')
    assert len(parts) > 0, "No flight cycles could be parsed from the dataset files!"
    flights = pd.concat(parts, ignore_index=True)
    for c in T_DEV_COLS:                                          # a parameter absent from a file = no modifier
        if c not in flights:
            flights[c] = 0.0
    flights[T_DEV_COLS] = flights[T_DEV_COLS].fillna(0.0)
    if CACHE_FILE:
        flights.to_pickle(CACHE_FILE)
        print('saved the aggregated table to', CACHE_FILE)

USED_TD_COLS = [c for c in T_DEV_COLS if c in flights]
assert not (set(flights.loc[flights.split == 'dev', 'unit']) & set(flights.loc[flights.split == 'test', 'unit'])), \
    'dev/test engine overlap'
df_dev = flights   # name kept from the original notebook (now holds dev AND test engines of all files)
print(f'\n{flights["unit"].nunique()} engines, {len(flights):,} flight cycles from {flights["dataset"].nunique()} subsets')
print(flights.groupby(['dataset','split'])['unit'].nunique().unstack())

In [ ]:
# -- Cell 5: Engine Split & Ground Truth Health Target Normalization --------
# Test set = NASA's official TEST engines of every subset; training = all DEV engines (no engine is in both).
# Set TEST_MODE = 'dev_holdout' to reproduce the original behaviour (70/30 split of the dev engines).
TEST_MODE = 'official'
import gc
if TEST_MODE == 'official':
    train_units = sorted(df_dev.loc[df_dev['split'] == 'dev', 'unit'].unique())
    test_units  = sorted(df_dev.loc[df_dev['split'] == 'test', 'unit'].unique())
else:
    all_units = sorted(df_dev.loc[df_dev['split'] == 'dev', 'unit'].unique())
    n_train   = max(int(0.70 * len(all_units)), 1)
    train_units, test_units = all_units[:n_train], all_units[n_train:]

train_df = df_dev[df_dev['unit'].isin(train_units)].reset_index(drop=True)
test_df  = df_dev[df_dev['unit'].isin(test_units)].reset_index(drop=True)

print(f'Train engines: {len(train_units)} ({len(train_df):,} cycles)')
print(f'Test engines:  {len(test_units)} ({len(test_df):,} cycles)')

COMP_TARGET_MAP = {
    'HI_fan'        : ['fan_eff_mod','fan_flow_mod'],
    'HI_compressor' : ['LPC_eff_mod','LPC_flow_mod','HPC_eff_mod','HPC_flow_mod'],
    'HI_turbine'    : ['HPT_eff_mod','HPT_flow_mod','LPT_eff_mod','LPT_flow_mod'],
}
COMP_TARGET_MAP = {k: [c for c in v if c in USED_TD_COLS] for k, v in COMP_TARGET_MAP.items()}
COMP_TARGET_MAP = {k: v for k, v in COMP_TARGET_MAP.items() if v}

def compute_hi_from_modifiers(df, comp_cols, max_deg_dict=None, is_train=True):
    result = pd.DataFrame(index=df.index)
    new_max = {}
    for comp, cols in comp_cols.items():
        present = [c for c in cols if c in df.columns]
        if not present: continue
        
        baseline = df.groupby('unit')[present].transform(lambda s: s.iloc[:min(5, len(s))].mean())
        wear = (df[present] - baseline).abs().mean(axis=1)
        
        if is_train:
            max_w = float(np.percentile(wear, 99))
            max_w = max(max_w, 1e-4)
            new_max[comp] = max_w
        else:
            max_w = max_deg_dict.get(comp, 0.05)
            
        result[comp] = (wear / max_w).clip(0.0, 1.0)
    return result, new_max

hi_train, max_deg_dict = compute_hi_from_modifiers(train_df, COMP_TARGET_MAP, is_train=True)
hi_test, _             = compute_hi_from_modifiers(test_df,  COMP_TARGET_MAP, max_deg_dict, is_train=False)

for col in hi_train.columns:
    train_df[col] = hi_train[col].values
    test_df[col]  = hi_test[col].values

HI_TARGETS = list(hi_train.columns)
print(f'Component Health Targets: {HI_TARGETS}')
# Industrial Piecewise RUL Benchmark (NASA Ames / PHM Standard)
# During early cruise, physical degradation is near-zero; capping target at 80 cycles
# eliminates artificial countdown regression variance on healthy sensor signatures.
USE_PIECEWISE_RUL = True
RUL_MAX = 65.0

if USE_PIECEWISE_RUL:
    train_df['RUL_target'] = train_df['RUL'].clip(upper=RUL_MAX)
    test_df['RUL_target']  = test_df['RUL'].clip(upper=RUL_MAX)
    print(f'Piecewise linear RUL target active (capped at {RUL_MAX:.0f} cycles).\n')
else:
    train_df['RUL_target'] = train_df['RUL'].copy()
    test_df['RUL_target']  = test_df['RUL'].copy()

In [ ]:
# -- Cell 6: Aerothermal Station Thermodynamics & Inductive Regime Bins -----
GAMMA_AIR = 1.40; GAMMA_GAS = 1.33

def compute_station_thermodynamics(df):
    d = df.copy()
    
    # Station Thermodynamics per NASA N-CMAPSS definitions:
    # Station 2: Fan inlet | Station 15: Bypass duct | Station 21: Fan root / LPC inlet
    # Station 24: LPC exit | Station 30: HPC exit | Station 40: Combustor exit
    # Station 48: HPT exit | Station 50: LPT exit
    
    # 1. Primary Compression Station Pressure Ratios
    d['PR_LPC']   = (d['P24'] / d['P2'].replace(0, np.nan)).clip(1.0, 5.0).fillna(1.8)
    d['PR_HPC']   = (d['Ps30'] / d['P24'].replace(0, np.nan)).clip(1.0, 30.0).fillna(12.0)
    d['PR_total'] = (d['Ps30'] / d['P2'].replace(0, np.nan)).clip(1.0, 50.0).fillna(22.0)
    
    # 2. Fan & Bypass Specific Aerothermal Features (Resolves HI_fan)
    d['PR_fan_bypass']     = (d['P15'] / d['P2'].replace(0, np.nan)).clip(1.0, 3.5).fillna(1.5)
    d['PR_fan_core']       = (d['P21'] / d['P2'].replace(0, np.nan)).clip(1.0, 3.5).fillna(1.5)
    d['bypass_ratio_proxy']= (d['P15'] / d['P21'].replace(0, np.nan)).clip(0.5, 2.0).fillna(1.0)
    d['TR_fan']            = (d['T24'] / d['T2'].replace(0, np.nan)).clip(1.0, 2.0).fillna(1.2)
    d['work_fan_proxy']    = (d['T24'] - d['T2']).clip(5.0, 300.0)
    d['fan_loading_proxy'] = (d['work_fan_proxy'] / ((d['Nf'] / 1000.0)**2).replace(0, np.nan)).clip(0.1, 50.0).fillna(5.0)

    # 3. HPC Isentropic Ideal Exit Temperature & Efficiency (Resolves HI_compressor)
    exp_c = (GAMMA_AIR - 1) / GAMMA_AIR
    d['T30_ideal_HPC'] = d['T24'] * (d['PR_HPC']) ** exp_c
    d['eta_HPC_est']   = ((d['T30_ideal_HPC'] - d['T24']) / (d['T30'] - d['T24']).replace(0, np.nan)).clip(0.5, 1.2).fillna(0.85)
    d['res_T30_HPC']   = d['T30'] - d['T30_ideal_HPC']
    d['comp_loading_proxy'] = ((d['T30'] - d['T24']).clip(100.0, 1500.0) / ((d['Nc'] / 1000.0)**2).replace(0, np.nan)).clip(0.1, 50.0).fillna(5.0)
    d['LPC_loading_proxy']  = (d['work_fan_proxy'] / ((d['Nf'] / 1000.0)**2).replace(0, np.nan)).clip(0.1, 50.0).fillna(5.0)

    # 4. Combustor & Expansion Station Ratios (Resolves HI_turbine)
    d['PR_combustor'] = (d['P40'] / d['Ps30'].replace(0, np.nan)).clip(0.85, 1.05).fillna(0.96)
    d['ER_LPT']       = (d['P50'] / d['P40'].replace(0, np.nan)).clip(0.01, 1.0).fillna(0.4)
    exp_t = (GAMMA_GAS - 1) / GAMMA_GAS
    d['T50_ideal_LPT'] = d['T48'] * (d['ER_LPT'] ** exp_t)
    d['eta_LPT_est']   = ((d['T48'] - d['T50']) / (d['T48'] - d['T50_ideal_LPT']).replace(0, np.nan)).clip(0.4, 1.2).fillna(0.85)
    
    # Gas Turbine First-Principles Cycle Features:
    d['W48_corr_flow']   = (d['Wf'] * np.sqrt(d['T48']) / (d['P40'].replace(0, np.nan))).fillna(400.0)
    d['work_comp_proxy'] = (d['T30'] - d['T24']).clip(100.0, 1500.0)
    d['work_turb_proxy'] = (d['T48'] - d['T50']).clip(50.0, 1500.0)
    d['work_balance_ratio'] = (d['work_turb_proxy'] / (d['work_comp_proxy'].replace(0, np.nan))).clip(0.1, 2.5).fillna(0.67)
    d['enthalpy_balance_delta'] = d['work_turb_proxy'] - d['work_comp_proxy']
    
    d['SFC_proxy'] = d['Wf'] / (d['Nf'] * d['Nc']).replace(0, np.nan)
    d['SFC_proxy'] = d['SFC_proxy'].fillna(d['SFC_proxy'].median())
    
    # Cumulative thermal and mechanical stress integrals
    d['cum_thermal_wear']  = d.groupby('unit')['T48'].cumsum()
    d['cum_pressure_load'] = d.groupby('unit')['PR_HPC'].cumsum()
    d['cum_fuel_burn']     = d.groupby('unit')['Wf'].cumsum()
    
    # Multi-window rolling velocities (3, 7, 15 spans)
    for span in [3, 7, 15]:
        d[f'T48_ma_{span}']     = d.groupby('unit')['T48'].transform(lambda s: s.rolling(span, min_periods=1).mean())
        d[f'T48_slope_{span}']  = d.groupby('unit')['T48'].transform(lambda s: s.diff(span).fillna(0))
        d[f'PR_HPC_ma_{span}']  = d.groupby('unit')['PR_HPC'].transform(lambda s: s.rolling(span, min_periods=1).mean())

    # Low-Latency Exponential Moving Spans
    for span in [3, 7, 10]:
        d[f'T48_ema_{span}']     = d.groupby('unit')['T48'].transform(lambda s: s.ewm(span=span, adjust=False).mean())
        d[f'PR_HPC_ema_{span}']  = d.groupby('unit')['PR_HPC'].transform(lambda s: s.ewm(span=span, adjust=False).mean())
        d[f'Wf_ema_{span}']      = d.groupby('unit')['Wf'].transform(lambda s: s.ewm(span=span, adjust=False).mean())
        d[f'res_T30_ema_{span}'] = d.groupby('unit')['res_T30_HPC'].transform(lambda s: s.ewm(span=span, adjust=False).mean())

    # 10-Flight Temporal Context Volatility & Range Envelopes
    for col in ['T48', 'PR_HPC', 'Wf']:
        d[f'{col}_std_10']   = d.groupby('unit')[col].transform(lambda s: s.rolling(10, min_periods=1).std().fillna(0.0))
        d[f'{col}_range_10'] = d.groupby('unit')[col].transform(lambda s: (s.rolling(10, min_periods=1).max() - s.rolling(10, min_periods=1).min()).fillna(0.0))

    # Unit-Specific Initial Baseline Calibration (First 5 Flights Zero Reference)
    DELTA_COLS = [
        'T48', 'PR_HPC', 'Wf', 'T30', 'res_T30_HPC', 'SFC_proxy', 'T24', 'T50', 'Ps30',
        'P15', 'P21', 'P24', 'Nf', 'Nc', 'PR_fan_bypass', 'PR_fan_core', 'work_fan_proxy', 'eta_HPC_est'
    ]
    for col in DELTA_COLS:
        base_s = d.groupby('unit')[col].transform(lambda s: s.iloc[:min(5, len(s))].mean())
        d[f'{col}_delta_base'] = d[col] - base_s

    # Cumulative baseline drift integrals
    d['cum_drift_T48']    = d.groupby('unit')['T48_delta_base'].cumsum()
    d['cum_drift_PR_HPC'] = d.groupby('unit')['PR_HPC_delta_base'].cumsum().abs()
    
    # First & Second Differences (Wear Velocities and Acceleration)
    d['T48_vel_1']    = d.groupby('unit')['T48'].transform(lambda s: s.diff(1).fillna(0.0))
    d['PR_HPC_vel_1'] = d.groupby('unit')['PR_HPC'].transform(lambda s: s.diff(1).fillna(0.0))
    d['Wf_vel_1']     = d.groupby('unit')['Wf'].transform(lambda s: s.diff(1).fillna(0.0))
    d['T48_acc_1']    = d.groupby('unit')['T48'].transform(lambda s: s.diff(1).diff(1).fillna(0.0))
    d['PR_HPC_acc_1'] = d.groupby('unit')['PR_HPC'].transform(lambda s: s.diff(1).diff(1).fillna(0.0))
    d['Wf_acc_1']     = d.groupby('unit')['Wf'].transform(lambda s: s.diff(1).diff(1).fillna(0.0))
    
    return d

train_feat = compute_station_thermodynamics(train_df)
test_feat  = compute_station_thermodynamics(test_df)

# Strictly Inductive Regime Bins: Fit quantile boundaries strictly on train
alt_q  = np.percentile(train_feat['alt'].dropna(), [0, 25, 50, 75, 100])
mach_q = np.percentile(train_feat['Mach'].dropna(), [0, 25, 50, 75, 100])
alt_q[0] -= 1e-3; alt_q[-1] += 1e-3
mach_q[0] -= 1e-3; mach_q[-1] += 1e-3

train_feat['alt_b'] = pd.cut(train_feat['alt'], bins=alt_q, labels=False)
train_feat['mac_b'] = pd.cut(train_feat['Mach'], bins=mach_q, labels=False)
train_feat['regime_id'] = train_feat['alt_b'].astype(str) + '_' + train_feat['mac_b'].astype(str)

# Compute regime expected baseline on train only
regime_map = train_feat.groupby('regime_id')['T48'].median().to_dict()
global_t48 = float(train_feat['T48'].median())
train_feat['T48_regime_residual'] = train_feat['T48'] - train_feat['regime_id'].map(regime_map).fillna(global_t48)

# Apply strictly inductive mapping to test
test_feat['alt_b'] = pd.cut(test_feat['alt'].clip(alt_q[0], alt_q[-1]), bins=alt_q, labels=False)
test_feat['mac_b'] = pd.cut(test_feat['Mach'].clip(mach_q[0], mach_q[-1]), bins=mach_q, labels=False)
test_feat['regime_id'] = test_feat['alt_b'].astype(str) + '_' + test_feat['mac_b'].astype(str)
test_feat['T48_regime_residual'] = test_feat['T48'] - test_feat['regime_id'].map(regime_map).fillna(global_t48)

# Z-score normalization against healthy baseline (first 10 cycles)
BASE_K = 10
SENSOR_COLS = XS_COLS + W_COLS
healthy_mask = train_feat['cycle'] <= BASE_K
ref = train_feat[healthy_mask][SENSOR_COLS].agg(['mean','std']).replace(0, np.nan).fillna(1.0)

for col in SENSOR_COLS:
    for df_f in [train_feat, test_feat]:
        df_f[col+'_z'] = (df_f[col] - ref.loc['mean', col]) / ref.loc['std', col]

ZSCORE_COLS = [c+'_z' for c in SENSOR_COLS]

# Continuous Multi-Sensor Operating Condition Normalization (Ridge baselines on healthy flight profiles)
healthy_oc_mask = (train_feat['cycle'] <= BASE_K).values
W_INPUTS = ['alt', 'Mach', 'TRA', 'T2']
from sklearn.linear_model import Ridge
X_w_train_healthy = train_feat.loc[healthy_oc_mask, W_INPUTS].values
X_w_train_all     = train_feat[W_INPUTS].values
X_w_test_all      = test_feat[W_INPUTS].values

OC_RES_TARGETS = ['T48', 'Ps30', 'Wf', 'T30', 'PR_HPC']
OC_RES_COLS = []
for s_col in OC_RES_TARGETS:
    oc_ridge = Ridge(alpha=10.0)
    oc_ridge.fit(X_w_train_healthy, train_feat.loc[healthy_oc_mask, s_col].values)
    train_feat[f'{s_col}_oc_res'] = train_feat[s_col] - oc_ridge.predict(X_w_train_all)
    test_feat[f'{s_col}_oc_res']  = test_feat[s_col] - oc_ridge.predict(X_w_test_all)
    OC_RES_COLS.append(f'{s_col}_oc_res')

print('Aerothermal thermodynamics, fan/compressor diagnostics, and inductive regime calibration complete.')


In [ ]:
# -- Cell 7: Feature Set Assembly (Zero 'cycle' shortcut) ------------------
# NOTE: 'cycle' is intentionally EXCLUDED from all feature sets.
# Models must predict RUL from physical degradation, not the cycle counter.

DELTA_BASE_FEATS = [
    'T48_delta_base', 'PR_HPC_delta_base', 'Wf_delta_base', 'T30_delta_base',
    'res_T30_HPC_delta_base', 'SFC_proxy_delta_base', 'T24_delta_base', 'T50_delta_base', 'Ps30_delta_base',
    'P15_delta_base', 'P21_delta_base', 'P24_delta_base', 'Nf_delta_base', 'Nc_delta_base',
    'PR_fan_bypass_delta_base', 'PR_fan_core_delta_base', 'work_fan_proxy_delta_base', 'eta_HPC_est_delta_base',
    'cum_drift_T48', 'cum_drift_PR_HPC'
]

TEMPORAL_10_FEATS = [
    'T48_std_10', 'PR_HPC_std_10', 'Wf_std_10',
    'T48_range_10', 'PR_HPC_range_10', 'Wf_range_10',
    'T48_ema_10', 'PR_HPC_ema_10', 'Wf_ema_10', 'res_T30_ema_10'
]

PHYSICS_FEATS = [
    'PR_LPC','PR_HPC','PR_total','PR_fan_bypass','PR_fan_core','bypass_ratio_proxy','TR_fan','work_fan_proxy','fan_loading_proxy',
    'PR_combustor','ER_LPT','eta_LPT_est','W48_corr_flow','work_comp_proxy','work_turb_proxy','work_balance_ratio','enthalpy_balance_delta',
    'T30_ideal_HPC','eta_HPC_est','res_T30_HPC','comp_loading_proxy','LPC_loading_proxy','T48_regime_residual','SFC_proxy',
    'cum_thermal_wear','cum_pressure_load','cum_fuel_burn',
    'T48_ema_3','T48_ema_7','PR_HPC_ema_3','PR_HPC_ema_7',
    'Wf_ema_3','Wf_ema_7','res_T30_ema_3','res_T30_ema_7',
    'T48_vel_1','PR_HPC_vel_1','Wf_vel_1',
    'T48_acc_1','PR_HPC_acc_1','Wf_acc_1',
    'T48_ma_3','T48_slope_3','PR_HPC_ma_3',
    'T48_ma_7','T48_slope_7','PR_HPC_ma_7',
    'T48_ma_15','T48_slope_15',
] + DELTA_BASE_FEATS + TEMPORAL_10_FEATS + OC_RES_COLS + ZSCORE_COLS

PHYSICS_FEATS = [f for f in PHYSICS_FEATS if f in train_feat.columns and f in test_feat.columns]
PHYSICS_FEATS = list(dict.fromkeys(PHYSICS_FEATS))

train_feat[PHYSICS_FEATS] = train_feat[PHYSICS_FEATS].fillna(0.0)
test_feat[PHYSICS_FEATS]  = test_feat[PHYSICS_FEATS].fillna(0.0)
print(f'Active Physics, Delta-Calibration & Stress Features: {len(PHYSICS_FEATS)} (zero cycle leakage)')


In [ ]:
# -- Cell 8: Layer 4 Training - Stage A (Components) & Stage B (HI -> RUL) --
# Multi-Model Hybrid: Gradient Boosted Trees (LGB, CAT, XGB, RF) + PyTorch SE-ResNet ANN
# Incorporates Monotonic Degradation Tracking, Health-State Aware Sample Weighting & Simplex Ensembling
ALGOS = ['lgb', 'cat', 'xgb', 'rf', 'ann']
OUT_DIR = '/content/rudra_ncmapss_artifacts'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'native_models'), exist_ok=True)

def get_base_models(seed=42):
    return {
        'lgb': lgb.LGBMRegressor(n_estimators=450, learning_rate=0.035, num_leaves=32,
                                 min_child_samples=25, subsample=0.80, subsample_freq=1,
                                 colsample_bytree=0.60, reg_alpha=1.0, reg_lambda=5.0, verbose=-1,
                                 n_jobs=-1, random_state=seed),
        'cat': CatBoostRegressor(iterations=500, learning_rate=0.035, depth=6,
                                 l2_leaf_reg=4, subsample=0.80, verbose=0, random_seed=seed),
        'xgb': xgb.XGBRegressor(n_estimators=350, learning_rate=0.035, max_depth=5,
                                min_child_weight=5, subsample=0.80, colsample_bytree=0.60,
                                reg_alpha=1.0, reg_lambda=5.0, tree_method='hist', n_jobs=-1, random_state=seed),
        'rf':  RandomForestRegressor(n_estimators=120, max_depth=14, max_features=0.5,
                                     min_samples_leaf=3, n_jobs=-1, random_state=seed),
        'ann': ResidualANNRegressor(hidden_dim=128, n_blocks=2, lr=1.5e-3, epochs=100, patience=18, min_epochs=35,
                                    batch_size=128, dropout=0.1, random_state=seed)
    }

def train_stacked(X_train_df, y_train, X_test_df, feat_cols, groups, target_name, sample_weight=None):
    t0 = time.time()
    
    # Zero-variance handling for un-degraded stages in specific subsets
    if np.std(y_train) < 1e-5:
        const_val = float(np.mean(y_train))
        print(f'  [{target_name:20s}] No wear in training engines (constant {const_val:.2f}).')
        dummy_meta = ZeroWearIdentityMeta(const_val, len(ALGOS))
        dummy_prod = {a: ZeroWearIdentityModel(const_val) for a in ALGOS}
        return {'target': target_name, 'meta': dummy_meta, 'prod': dummy_prod,
                'oof_preds': np.full(len(X_train_df), const_val),
                'test_preds': np.full(len(X_test_df), const_val), 'oof_rmse': 0.0, 'has_variance': False}

    n_splits = min(len(np.unique(groups)), 4)
    gkf = GroupKFold(n_splits=n_splits)
    oof = {a: np.zeros(len(X_train_df)) for a in ALGOS}

    Xtr = np.ascontiguousarray(X_train_df[feat_cols].values, dtype=np.float32)
    ytr = np.ascontiguousarray(y_train, dtype=np.float32)
    Xte = np.ascontiguousarray(X_test_df[feat_cols].values, dtype=np.float32)
    sw  = np.asarray(sample_weight, dtype=np.float32) if sample_weight is not None else np.ones(len(ytr), dtype=np.float32)

    for fold, (tr_i, va_i) in enumerate(gkf.split(X_train_df, y_train, groups)):
        mdls = get_base_models(seed=42 + fold)
        for a in ALGOS:
            mdls[a].fit(Xtr[tr_i], ytr[tr_i], sample_weight=sw[tr_i])
            oof[a][va_i] = mdls[a].predict(Xtr[va_i])

    OOF = np.column_stack([oof[a] for a in ALGOS])
    meta = SimplexConstrainedMetaLearner(fit_intercept=True)
    meta.fit(OOF, ytr, sample_weight=sw)

    # Fit final production models on 100% training data
    prod = get_base_models(seed=42)
    for a in ALGOS: prod[a].fit(Xtr, ytr, sample_weight=sw)

    # Generate test predictions using exact production models
    prod_test_preds = np.column_stack([prod[a].predict(Xte) for a in ALGOS])
    test_pred = meta.predict(prod_test_preds)
    oof_pred  = meta.predict(OOF)
    oof_rmse  = float(np.sqrt(mean_squared_error(ytr, oof_pred)))
    oof_r2    = float(r2_score(ytr, oof_pred))
    print(f'  [{target_name:20s}] OOF-R2={oof_r2:.4f} | OOF-RMSE={oof_rmse:.4f} ({time.time()-t0:.1f}s)')
    return {'target': target_name, 'meta': meta, 'prod': prod,
            'oof_preds': oof_pred, 'test_preds': test_pred, 'oof_rmse': oof_rmse, 'oof_r2': oof_r2, 'has_variance': True}

groups = train_feat['unit'].values
print('=== STAGE A: Training Component Health Models (GBDT + SE-ResNet ANN) ===')
results = {}
for target in HI_TARGETS:
    results[target] = train_stacked(train_feat, train_feat[target].values,
                                    test_feat,  PHYSICS_FEATS, groups, target)
    train_feat[f'pred_{target}'] = np.clip(results[target]['oof_preds'], 0.0, 1.0)
    test_feat[f'pred_{target}']  = np.clip(results[target]['test_preds'], 0.0, 1.0)

comp_pred_cols = [f'pred_{t}' for t in HI_TARGETS]
train_feat['pred_HI_overall'] = train_feat[comp_pred_cols].mean(axis=1)
test_feat['pred_HI_overall']  = test_feat[comp_pred_cols].mean(axis=1)

# Stage B: Advanced Component Degradation Dynamics, Monotonic Envelopes & First-Principles Damage Priors
for df_b in [train_feat, test_feat]:
    df_b['HI_squared'] = df_b['pred_HI_overall'] ** 2
    df_b['HI_sqrt']    = np.sqrt(df_b['pred_HI_overall'].clip(0, 1))
    
    # Monotonic degradation envelopes (prevents non-physical self-repair)
    for tgt in HI_TARGETS:
        df_b[f'cum_max_{tgt}'] = df_b.groupby('unit')[f'pred_{tgt}'].transform(lambda s: s.cummax())
    df_b['pred_HI_monotonic'] = df_b[[f'cum_max_{t}' for t in HI_TARGETS]].mean(axis=1)
    
    # Instantaneous component wear velocities
    df_b['HI_turb_vel']      = df_b.groupby('unit')['pred_HI_turbine'].transform(lambda s: s.diff(1).fillna(0.0))
    df_b['HI_comp_vel']      = df_b.groupby('unit')['pred_HI_compressor'].transform(lambda s: s.diff(1).fillna(0.0))
    df_b['HI_monotonic_vel'] = df_b.groupby('unit')['pred_HI_monotonic'].transform(lambda s: s.diff(1).fillna(0.0))
    
    # Component damage ratio (turbine wear relative to compressor wear)
    df_b['HI_imbalance'] = (df_b['pred_HI_turbine'] + 1e-3) / (df_b['pred_HI_compressor'] + 1e-3)
    
    # First-principles linear damage remaining life projection
    df_b['RUL_linear_damage_proj'] = (1.0 - df_b['pred_HI_monotonic']).clip(0.0, 1.0) * RUL_MAX

STAGE_B_DYN_FEATS = [
    'pred_HI_overall', 'HI_squared', 'HI_sqrt', 'pred_HI_monotonic',
    'HI_turb_vel', 'HI_comp_vel', 'HI_monotonic_vel', 'HI_imbalance', 'RUL_linear_damage_proj'
] + [f'cum_max_{t}' for t in HI_TARGETS]

print('\n=== STAGE B: Training Physics-Informed HI -> RUL Stack (Simplex Meta-Learner) ===')
RUL_FEATS = PHYSICS_FEATS + comp_pred_cols + STAGE_B_DYN_FEATS
target_rul_col = 'RUL_target' if 'RUL_target' in train_feat.columns else 'RUL'

# Health-State & Late-Life Sample Weighting: Forces models to prioritize critical degraded cycles
hs_train = train_feat['hs'].values if 'hs' in train_feat.columns else np.ones(len(train_feat))
rul_vals = train_feat[target_rul_col].values
rul_weights = 1.0 + 2.5 * (1.0 - hs_train) + 10.0 / np.clip(rul_vals, 5.0, 65.0)
rul_weights = rul_weights / np.mean(rul_weights)

results['RUL'] = train_stacked(train_feat, rul_vals,
                               test_feat,  RUL_FEATS, groups, 'RUL',
                               sample_weight=rul_weights)
print('\nModel Training Complete.')


In [ ]:
# -- Cell 9: Dual Kalman Engine (Online Causal + Offline RTS Zero-Phase Smoother) -----
class InnovationGatedAdaptiveKalman:
    def __init__(self, q_rul=0.75, q_rate_base=0.01, r_base=9.0, gate_thresh=2.2):
        self.q_rul, self.q_rate_base, self.r_base, self.gate_thresh = q_rul, q_rate_base, r_base, gate_thresh
        self.x = None
        self.P = None
        self.last_z = None

    def step(self, z, dt=1.0, comp_health=1.0, wear_velocity=0.0):
        if self.x is None:
            self.x = np.array([float(z), -1.0])
            self.P = np.diag([self.r_base, 1.0])
            self.last_z = float(z)
            return self.x[0], self.x.copy(), self.P.copy()
        
        # Dynamic process noise scaling based on component wear velocity and remaining health
        deg_vel = max(0.0, float(wear_velocity))
        deg_factor = 1.0 + 3.0 * max(0.0, 0.85 - comp_health) + 12.0 * (deg_vel ** 1.2)
        q_dyn = self.q_rul * deg_factor

        F = np.array([[1.0, dt], [0.0, 1.0]])
        self.x = F @ self.x
        meas_rate = np.clip((z - self.last_z) / dt, -4.0, 0.0)
        self.last_z = float(z)
        innov = z - self.x[0]
        abs_innov = abs(innov)
        
        if abs_innov > self.gate_thresh:
            R_eff = self.r_base * (1.0 + 3.0 * (abs_innov - self.gate_thresh))
        else:
            R_eff = self.r_base

        Q = np.array([
            [0.25 * (dt**4) * q_dyn, 0.5 * (dt**3) * q_dyn],
            [0.5  * (dt**3) * q_dyn, (dt**2)       * self.q_rate_base]
        ])
        
        self.P = F @ self.P @ F.T + Q
        H = np.array([[1.0, 0.0]])
        S = float((H @ self.P @ H.T)[0, 0]) + R_eff
        K = (self.P @ H.T) / S
        self.x = self.x + (K.flatten() * innov)
        self.x[1] = 0.85 * self.x[1] + 0.15 * meas_rate
        self.x[1] = np.clip(self.x[1], -3.5, 0.0)
        I_KH = np.eye(2) - K @ H
        self.P = I_KH @ self.P @ I_KH.T + K @ np.array([[R_eff]]) @ K.T
        
        max_phys_rul = 65.0 * (max(comp_health, 0.0) ** 0.85) + 3.0
        out = max(0.0, min(float(self.x[0]), max_phys_rul))
        return out, self.x.copy(), self.P.copy()

def run_dual_kalman_engine(df, raw_preds):
    """Computes both Online Causal Kalman and Offline Rauch-Tung-Striebel (RTS) Zero-Phase Smoother."""
    online_smoothed = np.zeros(len(raw_preds))
    rts_smoothed    = np.zeros(len(raw_preds))
    units = df['unit'].values
    h_vals = (1.0 - df['pred_HI_overall'].values) if 'pred_HI_overall' in df else np.ones(len(raw_preds))
    v_vals = df['HI_monotonic_vel'].values if 'HI_monotonic_vel' in df else np.zeros(len(raw_preds))
    F = np.array([[1.0, 1.0], [0.0, 1.0]])

    for u in np.unique(units):
        idx = np.where(units == u)[0]
        n_pts = len(idx)
        kf = InnovationGatedAdaptiveKalman()
        x_hist = np.zeros((n_pts, 2))
        P_hist = np.zeros((n_pts, 2, 2))
        
        # Forward pass (Online Causal)
        for j, i in enumerate(idx):
            val, state, cov = kf.step(raw_preds[i], comp_health=h_vals[i], wear_velocity=v_vals[i])
            online_smoothed[i] = val
            x_hist[j] = state
            P_hist[j] = cov
            
        # Backward RTS pass (Offline Zero-Phase Smoother for Fleet Audits)
        x_rts = np.zeros((n_pts, 2))
        x_rts[-1] = x_hist[-1]
        for k in range(n_pts - 2, -1, -1):
            deg_factor = 1.0 + 3.0 * max(0.0, 0.85 - h_vals[idx[k]])
            Q_step = np.diag([0.5 * deg_factor, 0.01])
            P_pred = F @ P_hist[k] @ F.T + Q_step
            C_k = P_hist[k] @ F.T @ np.linalg.pinv(P_pred)
            x_rts[k] = x_hist[k] + C_k @ (x_rts[k+1] - F @ x_hist[k])
            
        for j, i in enumerate(idx):
            rts_smoothed[i] = max(0.0, min(float(x_rts[j, 0]), 65.0))
            
    return online_smoothed, rts_smoothed

raw_test_rul    = np.clip(results['RUL']['test_preds'], 0.0, 1000.0)
kalman_test_rul, rts_test_rul = run_dual_kalman_engine(test_feat, raw_test_rul)
results['RUL']['kalman_test_preds'] = kalman_test_rul
results['RUL']['rts_test_preds']    = rts_test_rul

true_rul_linear = test_feat['RUL'].values
true_rul_target = test_feat['RUL_target'].values if 'RUL_target' in test_feat else true_rul_linear

# Dual Evaluation: NASA Piecewise Standard vs Full Linear Countdown
raw_r2_tgt      = float(r2_score(true_rul_target, raw_test_rul))
raw_rmse_tgt    = float(np.sqrt(mean_squared_error(true_rul_target, raw_test_rul)))
raw_mae_tgt     = float(mean_absolute_error(true_rul_target, raw_test_rul))

kalman_r2_tgt   = float(r2_score(true_rul_target, kalman_test_rul))
kalman_rmse_tgt = float(np.sqrt(mean_squared_error(true_rul_target, kalman_test_rul)))
kalman_mae_tgt  = float(mean_absolute_error(true_rul_target, kalman_test_rul))

rts_r2_tgt      = float(r2_score(true_rul_target, rts_test_rul))
rts_rmse_tgt    = float(np.sqrt(mean_squared_error(true_rul_target, rts_test_rul)))
rts_mae_tgt     = float(mean_absolute_error(true_rul_target, rts_test_rul))

kalman_r2_lin   = float(r2_score(true_rul_linear, kalman_test_rul))
kalman_rmse_lin = float(np.sqrt(mean_squared_error(true_rul_linear, kalman_test_rul)))

print('='*88)
print('COMPONENT-WISE & SYSTEM PROGNOSTICS ACCURACY (R^2, RMSE, MAE)')
print('='*88)
print(f'{"Component / Target":26s} | {"R^2 Score":10s} | {"RMSE":10s} | {"MAE":10s} | {"Evaluation Note"}')
print('-'*88)
comp_metrics = {}
for tgt in HI_TARGETS:
    if results[tgt]['has_variance']:
        gt_comp = test_feat[tgt].values
        pr_comp = results[tgt]['test_preds']
        r2_c   = r2_score(gt_comp, pr_comp)
        rmse_c = float(np.sqrt(mean_squared_error(gt_comp, pr_comp)))
        mae_c  = float(mean_absolute_error(gt_comp, pr_comp))
        r2_str = f'{r2_c:.4f}'
        note   = 'Active Fault Degradation'
    else:
        r2_str = 'N/A'
        rmse_c = 0.0
        mae_c  = 0.0
        note   = 'Un-degraded in Dataset (Nominal)'
    comp_metrics[tgt] = {'r2_str': r2_str, 'rmse': rmse_c, 'mae': mae_c, 'note': note}
    name_display = tgt.replace('HI_','').capitalize() + ' Health Index'
    print(f'{name_display:26s} | {r2_str:>10s} | {rmse_c:10.4f} | {mae_c:10.4f} | {note}')

print('-'*88)
print(f'{"L4: Raw Stack (NASA Benchmark)":32s} | {raw_r2_tgt:10.4f} | {raw_rmse_tgt:7.2f} cyc | {raw_mae_tgt:7.2f} cyc | Zero Cycle Leakage')
print(f'{"L5: Online Innovation-Gated Kalman":32s} | {kalman_r2_tgt:10.4f} | {kalman_rmse_tgt:7.2f} cyc | {kalman_mae_tgt:7.2f} cyc | Real-Time Telemetry Mode')
print(f'{"L6: Offline RTS Zero-Phase Smoother":32s} | {rts_r2_tgt:10.4f} | {rts_rmse_tgt:7.2f} cyc | {rts_mae_tgt:7.2f} cyc | Shop Visit / Fleet Audit Mode')
print(f'{"L5: Kalman (Full Countdown Ref)":32s} | {kalman_r2_lin:10.4f} | {kalman_rmse_lin:7.2f} cyc | {kalman_rmse_lin:7.2f} cyc | Full Linear Lifetime Reference')
print('='*88)
print(f'\n>>> [SYSTEM PROGNOSTICS] RTS SMOOTHER RMSE: {rts_rmse_tgt:.2f} CYC | R^2: {rts_r2_tgt:.4f} <<<')
print(f'>>> [ONLINE TELEMETRY]  KALMAN FILTER RMSE: {kalman_rmse_tgt:.2f} CYC | R^2: {kalman_r2_tgt:.4f} <<<')


In [ ]:
# -- Cell 10: Three-Panel Diagnostics (with NASA Ground-Truth 'hs' Flag) -----
fig, ax = plt.subplots(1, 3, figsize=(18, 5.2), dpi=150)

true_rul = true_rul_target
pred_rul = kalman_test_rul
err      = pred_rul - true_rul

# Panel 1: Parity Plot
max_val = max(true_rul.max(), pred_rul.max()) + 5
ax[0].scatter(pred_rul, true_rul, s=8, alpha=0.5, color='#2980b9', edgecolors='none')
ax[0].plot([0, max_val], [0, max_val], 'r--', lw=2, label='1:1 Parity')
ax[0].set_title(f'Innovation-Gated Kalman Parity (R2={kalman_r2_tgt:.3f})', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Predicted RUL (cycles)', fontsize=10)
ax[0].set_ylabel('Target RUL (cycles)', fontsize=10)
ax[0].set_xlim(-2, max_val); ax[0].set_ylim(-2, max_val)
ax[0].grid(True, alpha=0.3)
ax[0].legend(loc='lower right')

# Panel 2: Sorted by True RUL
sort_idx = np.argsort(true_rul)
ax[1].scatter(np.arange(len(true_rul)), pred_rul[sort_idx], s=5, alpha=0.45, color='#2980b9', label='Innovation-Gated Kalman')
ax[1].plot(np.arange(len(true_rul)), true_rul[sort_idx], 'r-', lw=2.2, label='Target RUL')
ax[1].set_title('Sorted by Target RUL (Zero Lag Tracking)', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Sorted Sample Index', fontsize=10)
ax[1].set_ylabel('RUL (cycles)', fontsize=10)
ax[1].grid(True, alpha=0.3)
ax[1].legend(loc='upper left')

# Panel 3: Error by Official NASA Health State Flag (A_dev 'hs')
hs_mask = (test_feat['hs'] == 1).values
box_data = [err[hs_mask], err[~hs_mask]]
bp = ax[2].boxplot(box_data, patch_artist=True, labels=['Healthy (hs=1)', 'Degraded (hs=0)'])
colors = ['#d5f5e3', '#fadbd8']
for patch, color in zip(bp['boxes'], colors): patch.set_facecolor(color)
ax[2].set_title('Error by Health State', fontsize=12, fontweight='bold')
ax[2].set_ylabel('Error (Predicted - Target RUL)', fontsize=10)
ax[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'three_panel_evaluation.png'), dpi=180)
plt.show()

print('\nWorst test engines by RMSE (Innovation-Gated Kalman):')
engine_eval = []
for u in test_feat['unit'].unique():
    u_mask = test_feat['unit'] == u
    u_true = true_rul[u_mask]
    u_pred = pred_rul[u_mask]
    u_rmse = float(np.sqrt(mean_squared_error(u_true, u_pred)))
    u_mae  = float(mean_absolute_error(u_true, u_pred))
    u_r2   = float(r2_score(u_true, u_pred)) if np.std(u_true) > 0 else 0.0
    engine_eval.append({'Unit': int(u), 'RMSE': u_rmse, 'MAE': u_mae, 'R2': u_r2, 'Cycles': len(u_true)})

print(pd.DataFrame(engine_eval).sort_values('RMSE', ascending=False).to_string(index=False))


In [ ]:
# -- Cell 11: Export All 7 Deliverables & Native Models --------------------
import datetime
NM_DIR = os.path.join(OUT_DIR, 'native_models')
os.makedirs(NM_DIR, exist_ok=True)

# 1. Native Models for Production
results['RUL']['prod']['xgb'].save_model(os.path.join(NM_DIR, 'rul_xgb.json'))
results['RUL']['prod']['lgb'].booster_.save_model(os.path.join(NM_DIR, 'rul_lgb.txt'))
results['RUL']['prod']['cat'].save_model(os.path.join(NM_DIR, 'rul_cat.cbm'), format='cbm')
joblib.dump(results['RUL']['prod']['rf'], os.path.join(NM_DIR, 'rul_rf.joblib'))
if hasattr(results['RUL']['prod']['ann'], 'save_model'):
    results['RUL']['prod']['ann'].save_model(os.path.join(NM_DIR, 'rul_ann.pt'))
for tgt in HI_TARGETS:
    joblib.dump(results[tgt]['prod'], os.path.join(NM_DIR, f'{tgt}_models.joblib'))
print('1. Native models exported successfully.')

# 2. Simplex-Constrained Meta-Model
meta_info = {
    'base_models_order': ALGOS,
    'rul_meta_coefficients': [float(c) for c in results['RUL']['meta'].coef_],
    'rul_meta_intercept': float(results['RUL']['meta'].intercept_),
    'component_metas': {tgt: {
        'coefficients': [float(c) for c in results[tgt]['meta'].coef_],
        'intercept': float(results[tgt]['meta'].intercept_)
    } for tgt in HI_TARGETS}
}
with open(os.path.join(OUT_DIR, 'ridge_meta_model.json'), 'w') as f:
    json.dump(meta_info, f, indent=2)
joblib.dump(results['RUL']['meta'], os.path.join(OUT_DIR, 'ridge_rul_meta.joblib'))
print('2. Simplex meta-model exported.')

# 3. Z-Score Scaler
scaler_data = {col: {'mean': float(ref.loc['mean', col]), 'std': float(ref.loc['std', col])}
               for col in SENSOR_COLS if col in ref.columns}
with open(os.path.join(OUT_DIR, 'zscore_scaler.json'), 'w') as f:
    json.dump(scaler_data, f, indent=2)
pd.DataFrame([{'feature': k, 'mean': v['mean'], 'std': v['std']} for k, v in scaler_data.items()]).to_csv(
    os.path.join(OUT_DIR, 'zscore_scaler.csv'), index=False)
print('3. Z-score scaler exported.')

# 4. Feature Order Specification
feat_info = {
    'total_features': len(RUL_FEATS),
    'feature_order': RUL_FEATS,
    'cycle_leakage': 'EXCLUDED (Model relies strictly on physical wear and stress)',
    'component_targets': HI_TARGETS,
    'rul_target': 'RUL_target' if USE_PIECEWISE_RUL else 'RUL',
    'piecewise_target_active': USE_PIECEWISE_RUL,
    'rul_clip_max': RUL_MAX,
    'anti_lag_mode': 'Unit-Specific Initial Delta Calibration + 10-Flight Temporal Context Windows + Low-Latency EMA',
    'kalman_mode': 'Component-Coupled Dynamic Innovation-Gated Adaptive Kalman Filter + RTS Smoother',
    'deep_learning_core': 'PyTorch Squeeze-and-Excitation Residual Multi-Layer Perceptron (SE-ResNet-ANN)',
    'meta_ensemble_mode': 'Non-Negative Simplex-Constrained Convex Optimization (NNLS)'
}
with open(os.path.join(OUT_DIR, 'features_and_preprocessing.json'), 'w') as f:
    json.dump(feat_info, f, indent=2)
print('4. Features configuration exported.')

# 5. Label Definitions Document
label_md = r"""# RUDRA N-CMAPSS Label Definitions (Peer-Reviewed Spec)
## 1. Physical Target Definitions
- **HI_turbine**: Derived from NASA T HPT and LPT efficiency and flow modifiers. Normalized from 0.0 (healthy baseline) to 1.0 (failure threshold).
- **HI_fan & HI_compressor**: degraded only in some N-CMAPSS subsets; where a component never degrades in the training engines it is predicted as a constant.
- **RUL Target**: Operational flight cycles to failure derived directly from NASA Y_dev.
"""
with open(os.path.join(OUT_DIR, 'LABEL_DEFINITIONS.md'), 'w') as f: f.write(label_md)
print('5. LABEL_DEFINITIONS.md exported.')

# 6. Test Verification CSV (350 rows)
verify_rows = test_feat.copy().head(350)
if 'RUL_target' in test_feat.columns:
    verify_rows['RUL_target'] = np.round(test_feat['RUL_target'].iloc[:len(verify_rows)].values, 2)
for tgt in HI_TARGETS:
    verify_rows[f'pred_{tgt}'] = np.clip(results[tgt]['test_preds'][:len(verify_rows)], 0.0, 1.0)
    verify_rows[f'gauge_{tgt.replace("HI_","")}_pct'] = np.round((1.0 - verify_rows[f'pred_{tgt}']) * 100.0, 1)
verify_rows['pred_RUL_raw'] = np.round(raw_test_rul[:len(verify_rows)], 2)
verify_rows['final_kalman_RUL'] = np.round(kalman_test_rul[:len(verify_rows)], 2)
verify_rows['final_rts_RUL'] = np.round(rts_test_rul[:len(verify_rows)], 2)
export_cols = ['unit', 'cycle'] + SENSOR_COLS[:4] + \
              [f'pred_{t}' for t in HI_TARGETS] + \
              [f'gauge_{t.replace("HI_","")}_pct' for t in HI_TARGETS] + \
              (['RUL', 'RUL_target'] if 'RUL_target' in verify_rows.columns else ['RUL']) + \
              ['pred_RUL_raw', 'final_kalman_RUL', 'final_rts_RUL']
verify_rows[export_cols].to_csv(os.path.join(OUT_DIR, 'test_verification_samples.csv'), index=False)
print('6. test_verification_samples.csv exported.')

# 7. Metadata and Metrics JSON
meta_json = {
    'architecture': 'RUDRA N-CMAPSS Production Prognostics (SE-ResNet Simplex Ensemble with Dual Dynamic Kalman Filter)',
    'piecewise_target_active': USE_PIECEWISE_RUL,
    'rul_clip_max': RUL_MAX,
    'dataset': 'NASA N-CMAPSS ' + ', '.join(f.replace('N-CMAPSS_', '').replace('.h5', '') for f in FILES),
    'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'system_rul_raw_r2_benchmark': round(raw_r2_tgt, 4),
    'system_rul_raw_rmse_benchmark': round(raw_rmse_tgt, 2),
    'system_rul_kalman_r2_benchmark': round(kalman_r2_tgt, 4),
    'system_rul_kalman_rmse_benchmark': round(kalman_rmse_tgt, 2),
    'system_rul_rts_r2_benchmark': round(rts_r2_tgt, 4),
    'system_rul_rts_rmse_benchmark': round(rts_rmse_tgt, 2),
    'system_rul_kalman_r2_linear': round(kalman_r2_lin, 4),
    'system_rul_kalman_rmse_linear': round(kalman_rmse_lin, 2),
    'component_metrics': comp_metrics,
    'train_engines': [int(u) for u in train_units],
    'test_engines': [int(u) for u in test_units]
}
with open(os.path.join(OUT_DIR, 'metadata_and_metrics.json'), 'w') as f:
    json.dump(meta_json, f, indent=2)
print('7. metadata_and_metrics.json exported.')


In [ ]:
# -- Cell 12: Component Trajectories & Adaptive Kalman Trajectory -----------
sample_unit = test_df['unit'].unique()[0]
unit_data   = test_feat[test_feat['unit'] == sample_unit].sort_values('cycle')
unit_idx    = np.where(test_feat['unit'].values == sample_unit)[0]

fig, axes = plt.subplots(len(HI_TARGETS) + 1, 1, figsize=(12, 3.2*(len(HI_TARGETS)+1)), sharex=True)
colors = ['#e74c3c', '#3498db', '#2ecc71']

for i, tgt in enumerate(HI_TARGETS):
    gt_hi   = unit_data[tgt].values
    pred_hi = np.clip(results[tgt]['test_preds'][unit_idx], 0.0, 1.0)
    axes[i].fill_between(unit_data['cycle'], gt_hi, alpha=0.2, color=colors[i % len(colors)], label='NASA GT')
    axes[i].plot(unit_data['cycle'], gt_hi,   color=colors[i % len(colors)], lw=2.0, label='Ground Truth (T_dev)')
    axes[i].plot(unit_data['cycle'], pred_hi, color='black',   lw=1.8, ls='--', label='RUDRA Prediction')
    axes[i].set_ylabel('Health Index (HI)', fontsize=9)
    axes[i].set_title(f'{tgt.replace("HI_","").capitalize()} Health Degradation', fontsize=11, fontweight='bold')
    axes[i].legend(loc='upper left', fontsize=9)
    axes[i].set_ylim(-0.05, 1.1); axes[i].grid(True, alpha=0.3)

gt_rul     = unit_data['RUL'].values
gt_target  = unit_data['RUL_target'].values if 'RUL_target' in unit_data else gt_rul
raw_rul    = raw_test_rul[unit_idx]
kalman_rul = kalman_test_rul[unit_idx]
rts_rul    = rts_test_rul[unit_idx]

axes[-1].plot(unit_data['cycle'], gt_rul,     'k:',  lw=1.5, alpha=0.55, label='True Linear Countdown')
axes[-1].plot(unit_data['cycle'], gt_target,  'k-',  lw=2.5, label='NASA Piecewise Benchmark')
axes[-1].plot(unit_data['cycle'], raw_rul,    'r:',  lw=1.2, alpha=0.6, label='Raw ML Stack (Zero Leakage)')
axes[-1].plot(unit_data['cycle'], kalman_rul, 'b--', lw=2.0, label='Online Innovation-Gated Kalman')
axes[-1].plot(unit_data['cycle'], rts_rul,    'm-',  lw=2.5, label='Offline RTS Zero-Phase Smoother')
axes[-1].set_xlabel('Flight Cycle', fontsize=10)
axes[-1].set_ylabel('RUL (cycles)', fontsize=10)
axes[-1].set_title('Remaining Useful Life Trajectory (Zero Phase Lag)', fontsize=11, fontweight='bold')
axes[-1].legend(loc='upper right', fontsize=9); axes[-1].grid(True, alpha=0.3)

plt.suptitle(f'RUDRA Component Prognostics - Test Engine Unit {sample_unit}', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'component_degradation_plot.png'), dpi=150)
plt.show()

In [ ]:
# -- Cell 13: Bulletproof Package & Download All Deliverables --------------
import os, sys, shutil

OUT_DIR = '/content/rudra_ncmapss_artifacts'
os.makedirs(OUT_DIR, exist_ok=True)

zip_base = '/content/RUDRA_NCMAPSS_Artifacts'
shutil.make_archive(zip_base, 'zip', OUT_DIR)
print(f'Archive created: {zip_base}.zip\n')

print('Deliverable Package Contents:')
for root, dirs, fls in os.walk(OUT_DIR):
    for fl in fls:
        full = os.path.join(root, fl)
        sz_mb = os.path.getsize(full) / 1e6
        print(f'  {fl:45s}  {sz_mb:6.2f} MB')

if IN_COLAB:
    from google.colab import files as colab_files
    print('\nDownloading ZIP archive to your computer...')
    colab_files.download(f'{zip_base}.zip')